# Bridge smoke test — phases-vyakarana.md V1

A hand-written `simulate.dfa` trace, pasted into a cell and rendered by the bundle in
`vyakarana/static/` — no Python package yet, the widget class is defined inline.
Build first: `pnpm -F @tape-n-trace/bridge build`. Evidence of the JupyterLab run is
recorded by executing this notebook there and committing the dated output.

In [1]:
import pathlib
import anywidget
import traitlets

STATIC = pathlib.Path.cwd().parent.parent / 'vyakarana' / 'vyakarana' / 'static'

class TntWidget(anywidget.AnyWidget):
    _esm = STATIC / 'widget.js'
    _css = STATIC / 'widget.css'
    payload = traitlets.Dict(allow_none=True, default_value=None).tag(sync=True)
    trace = traitlets.Dict(allow_none=True, default_value=None).tag(sync=True)
    step = traitlets.Int(0).tag(sync=True)
    options = traitlets.Dict().tag(sync=True)

In [2]:
MACHINE = {
    'kind': 'DFA',
    'states': ['a', 'b'],
    'alphabet': ['0', '1'],
    'transitions': [
        {'id': 't1', 'from': 'a', 'read': '0', 'to': 'b'},
        {'id': 't2', 'from': 'b', 'read': '1', 'to': 'b'},
    ],
    'start': 'a',
    'accepting': ['b'],
}

TRACE = {
    'kind': 'simulate.dfa',
    'engineVersion': '0.1.0',
    'input': {'word': '01'},
    'steps': [
        {'index': 0, 'narration': 'Start in state a with the whole input unread.',
         'highlight': [{'type': 'state', 'id': 'a', 'role': 'start'}],
         'snapshot': {'machine': MACHINE, 'input': ['0', '1'], 'position': 0, 'state': 'a', 'status': 'running'}},
        {'index': 1, 'narration': 'Read 0 and move to state b.',
         'highlight': [{'type': 'state', 'id': 'b', 'role': 'current'}, {'type': 'transition', 'id': 't1', 'role': 'taken'}],
         'snapshot': {'machine': MACHINE, 'input': ['0', '1'], 'position': 1, 'state': 'b', 'status': 'running'}},
        {'index': 2, 'narration': 'Read 1, stay in b, and accept: b is an accepting state.',
         'highlight': [{'type': 'state', 'id': 'b', 'role': 'accepting'}],
         'snapshot': {'machine': MACHINE, 'input': ['0', '1'], 'position': 2, 'state': 'b', 'status': 'accepted'}},
    ],
    'result': {'type': 'acceptance', 'accepted': True},
    'meta': {'stepCount': 3, 'counters': {}},
}

w = TntWidget(trace=TRACE)
w

In [4]:
# Python drives the widget: the transport should jump to the last step…
w.step = 2
# …and reading it back returns what the widget shows.
w.step

2